# Notebook para treinamento e testes de modelos

In [1]:
import pandas as pd

import model_generators as mg
import utils

import datetime
from dateutil.relativedelta import relativedelta
import optuna
import logging

from dataclasses import dataclass

logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.WARNING)
optuna.logging.set_verbosity(optuna.logging.ERROR)

from enum import Enum
from __future__ import annotations

c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [2]:
@dataclass()
class Config():
    """Classe para a definicao das variaveis de ambiente"""
    horizonte_previsao: int
    tamanho_teste: int
    n_trials_skus: int
    metrica_erro: Status
    tolerancia_fipe: int
    tolerancia_exog: float
    skus: list[int]
    data_ref: datetime.datetime
    dt_inicio_teste: datetime.datetime
    dt_limite_inferior_exog: datetime.datetime
    exog_list: list[str]

    def __init__(self, horizonte_previsao, tamanho_teste, n_trials_skus, n_trials_exog, metrica_erro, tolerancia_fipe, tolerancia_exog, skus, data_ref, dt_limite_inferior_exog, exog_list):
        self.horizonte_previsao = horizonte_previsao
        self.tamanho_teste = tamanho_teste
        self.n_trials_skus = n_trials_skus
        self.n_trials_exog = n_trials_exog
        self.metrica_erro = metrica_erro
        self.tolerancia_fipe = tolerancia_fipe
        self.tolerancia_exog = tolerancia_exog
        self.skus = skus
        self.data_ref = data_ref
        self.dt_limite_inferior_exog = dt_limite_inferior_exog
        self.exog_list = exog_list
        self.dt_inicio_teste = data_ref + relativedelta(months=-tamanho_teste)

class Status(Enum):
    MAE = 'mae'
    MAPE = 'mape'

In [3]:
config = Config(
    horizonte_previsao = 3,
    tamanho_teste = 6,
    n_trials_skus = 5,
    n_trials_exog = 5,
    metrica_erro = Status.MAE,
    tolerancia_fipe = 200,
    tolerancia_exog = 0.01,
    skus = [100, 1832, 2134, 5112, 7023],
    data_ref = datetime.datetime(2026, 5, 1),
    dt_limite_inferior_exog = datetime.datetime(int(pd.to_datetime('today').year - 10), 1, 1),
    exog_list = ['valor', 'exchange_rate']
)

## Leitura da base de dados

In [4]:
# Base fipe historica
fipe_path = './data/dados_fipe_tratados.csv'
# Base da taxa de cambio
exchange_path = './data/DEXBZUS_tratados.csv'
# Base IPCA
ipca_path = './data/bcdata.sgs.433_tratados.csv' 

df_fipe = pd.read_csv(fipe_path).drop(columns=['Unnamed: 0', 'year_of_reference', 'month_of_reference'])

df_exchange = pd.read_csv(exchange_path)

df_ipca = pd.read_csv(ipca_path).drop(columns=['data', 'ipca'])

In [5]:
df_ipca['date'] = pd.to_datetime(df_ipca['date'])
df_exchange['date'] = pd.to_datetime(df_exchange['date'])
df_fipe['reference_date'] = pd.to_datetime(df_fipe['reference_date'], format='ISO8601')

df_ipca.index = df_ipca['date']
df_exchange.index = df_exchange['date']

In [6]:
# Criacao dos skus
# Refazer essa logica para ele ir incrementando
df_fipe['sku'] = df_fipe.groupby(['brand_name', 'model_name', 'fuel_name', 'year']).ngroup()

In [7]:
display(df_fipe.head())

,reference_date,brand_name,model_name,year,fuel_name,brl_price,sku
0,2021-01-01,Fiat,147 C/ CL,1987,Gasolina,2723.0,2
1,2021-01-01,Fiat,147 C/ CL,1986,Gasolina,2484.0,1
2,2021-01-01,Fiat,147 C/ CL,1985,Gasolina,2324.0,0
3,2021-01-01,Fiat,147 Furgão (todos),1987,Gasolina,2199.0,5
4,2021-01-01,Fiat,147 Furgão (todos),1986,Gasolina,2094.0,4


In [8]:
df_previsao_skus = utils.coletar_base_skus(config.data_ref, config.skus, df_fipe)

## Separar os dados em treino e teste para exógenas

In [9]:
df_exog_train, df_exog_test = utils.separar_treino_teste_exogena(df_ipca, df_exchange, config.dt_inicio_teste, config.dt_limite_inferior_exog)

## Modelos

#### Modelo do Câmbio

##### Prophet

In [10]:
df_prophet_train_exchange, df_prophet_test_exchange = utils.criar_dataset_prophet(df_exog_train, df_exog_test, 'date', 'exchange_rate', [])

model_exchange_prophet, info_exchange_prophet, best_value_exchange_prophet = mg.generate_prophet_model(
    df_prophet_train_exchange,
    df_prophet_test_exchange,
    [],
    config.n_trials_exog,
    config.metrica_erro,
    config.tolerancia_exog
)

model_exchange_prophet.fit(
    df_prophet_train_exchange,
)

forecast_exchange = model_exchange_prophet.predict(
    df_prophet_test_exchange[['ds']]
)

display(pd.concat([df_prophet_test_exchange['y'], forecast_exchange['yhat']], axis=1))


15:48:47 - cmdstanpy - INFO - Chain [1] start processing
15:48:47 - cmdstanpy - INFO - Chain [1] done processing
15:48:47 - cmdstanpy - INFO - Chain [1] start processing
15:48:47 - cmdstanpy - INFO - Chain [1] done processing
15:48:47 - cmdstanpy - INFO - Chain [1] start processing
15:48:47 - cmdstanpy - INFO - Chain [1] done processing
15:48:47 - cmdstanpy - INFO - Chain [1] start processing
15:48:47 - cmdstanpy - INFO - Chain [1] done processing
15:48:48 - cmdstanpy - INFO - Chain [1] start processing
15:48:48 - cmdstanpy - INFO - Chain [1] done processing
15:48:48 - cmdstanpy - INFO - Chain [1] start processing
15:48:48 - cmdstanpy - INFO - Chain [1] done processing


,y,yhat
0,5.341483,5.514070
1,5.455709,5.534694
2,5.331620,5.526998
3,5.198805,5.475101
4,5.229641,5.521034
5,5.033945,5.545783


##### SARIMAX

In [11]:
model_exchange_sarimax, info_exchange_sarimax, best_value_exchange_sarimax = mg.generate_sarimax_model(
    df_exog_train['exchange_rate'],
    df_exog_test['exchange_rate'],
    None,
    None,
    config.n_trials_exog,
    config.metrica_erro,
    config.tolerancia_exog
)

model_exchange_sarimax_fit = model_exchange_sarimax.fit(disp=False)

forecasts_exchange_sarimax = model_exchange_sarimax_fit.forecast(steps=len(df_exog_test['exchange_rate']))

display(pd.concat([forecasts_exchange_sarimax, df_exog_test['exchange_rate']], axis=1))

c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\sta

,predicted_mean,exchange_rate
2025-11-01,5.334591,5.341483
2025-12-01,5.343270,5.455709
2026-01-01,5.327556,5.331620
2026-02-01,5.266699,5.198805
2026-03-01,5.255125,5.229641
2026-04-01,5.236291,5.033945


#### Modelo do IPCA

##### Prophet

In [12]:
df_prophet_train_ipca, df_prophet_test_ipca = utils.criar_dataset_prophet(df_exog_train, df_exog_test, 'date', 'valor', [])

model_ipca_prophet, info_ipca_prophet, best_value_ipca_prophet = mg.generate_prophet_model(
    df_prophet_train_ipca,
    df_prophet_test_ipca,
    [],
    config.n_trials_exog,
    config.metrica_erro,
    config.tolerancia_exog
)

model_ipca_prophet.fit(
    df_prophet_train_ipca,
)

forecast_ipca = model_ipca_prophet.predict(
    df_prophet_test_ipca[['ds']]
)

display(pd.concat([df_prophet_test_ipca['y'], forecast_ipca['yhat']], axis=1))

15:48:53 - cmdstanpy - INFO - Chain [1] start processing
15:48:53 - cmdstanpy - INFO - Chain [1] done processing
15:48:53 - cmdstanpy - INFO - Chain [1] start processing
15:48:53 - cmdstanpy - INFO - Chain [1] done processing
15:48:53 - cmdstanpy - INFO - Chain [1] start processing
15:48:53 - cmdstanpy - INFO - Chain [1] done processing
15:48:53 - cmdstanpy - INFO - Chain [1] start processing
15:48:53 - cmdstanpy - INFO - Chain [1] done processing
15:48:54 - cmdstanpy - INFO - Chain [1] start processing
15:48:54 - cmdstanpy - INFO - Chain [1] done processing
15:48:54 - cmdstanpy - INFO - Chain [1] start processing
15:48:54 - cmdstanpy - INFO - Chain [1] done processing


,y,yhat
0,0.18,0.467615
1,0.33,0.653209
2,0.33,0.464323
3,0.70,0.714527
4,0.88,0.587832
5,0.67,0.469243


##### SARIMAX

In [13]:
model_ipca_sarimax, info_ipca_sarimax, best_value_ipca_sarimax = mg.generate_sarimax_model(
    df_exog_train['valor'],
    df_exog_test['valor'],
    None,
    None,
    config.n_trials_exog,
    config.metrica_erro,
    config.tolerancia_exog
)

model_ipca_sarimax_fit = model_ipca_sarimax.fit(disp=False)

forecasts_ipca_sarimax = model_ipca_sarimax_fit.forecast(steps=len(df_exog_test['valor']))

display(pd.concat([forecasts_ipca_sarimax, df_exog_test['valor']], axis=1))

c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Li

,predicted_mean,valor
2025-11-01,0.170489,0.18
2025-12-01,0.419152,0.33
2026-01-01,0.194075,0.33
2026-02-01,0.428439,0.70
2026-03-01,0.398575,0.88
2026-04-01,0.256522,0.67


#### Selecionar melhor forecast das exógenas

In [14]:
dict_kwargs = {
    'best_value_ipca_prophet': best_value_ipca_prophet,
    'best_value_ipca_sarimax': best_value_ipca_sarimax,
    'best_value_exchange_prophet': best_value_exchange_prophet,
    'best_value_exchange_sarimax': best_value_exchange_sarimax,
    'info_ipca_prophet': info_ipca_prophet,
    'info_ipca_sarimax': info_ipca_sarimax,
    'info_exchange_prophet': info_exchange_prophet,
    'info_exchange_sarimax': info_exchange_sarimax,
    'df_prophet_train_ipca': df_prophet_train_ipca,
    'df_prophet_test_ipca': df_prophet_test_ipca,
    'df_prophet_train_exchange': df_prophet_train_exchange,
    'df_prophet_test_exchange': df_prophet_test_exchange,
    'df_exog_train': df_exog_train,
    'df_exog_test': df_exog_test
}

df_exog_previsao, modelo_escolhido_ipca, modelo_escolhido_exchange = utils.selecionar_modelo_exog(
    horizonte_previsao=config.horizonte_previsao,
    **dict_kwargs
)

print(f"Modelo escolhido IPCA: {modelo_escolhido_ipca}")
print(f"Modelo escolhido taxa de câmbio: {modelo_escolhido_exchange}")
display(df_exog_previsao)

c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)


Modelo escolhido IPCA: SARIMAX
Modelo escolhido taxa de câmbio: SARIMAX


c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


,valor,exchange_rate
2026-05-01,0.459061,4.956257
2026-06-01,0.401069,4.913351
2026-07-01,0.286251,4.844246


#### Modelo da FIPE

In [19]:
previsoes_por_sku = {}
modelo_vencedor_por_sku = {}

for sku in config.skus:
    df_sku_atual = df_previsao_skus.query('sku == @sku')

    if df_sku_atual.empty:
        print(f"Pulando SKU: {sku} por poucos dados...")
        continue

    df_train_sku, df_test_sku = utils.separar_treino_teste_sku(df_sku_atual, config.data_ref, config.dt_inicio_teste)

    df_exog_train_sku = df_exog_train[df_exog_train.index.isin(df_train_sku['reference_date'])].copy()
    df_exog_test_sku = df_exog_test.copy()

    df_exog_train_sku = df_exog_train_sku[config.exog_list]
    df_exog_test_sku = df_exog_test_sku[config.exog_list]

    df_train_prophet_sku, df_test_prophet_sku = utils.criar_dataset_prophet(
        pd.concat([df_train_sku, df_exog_train_sku], axis=1),
        pd.concat([df_test_sku, df_exog_test], axis=1),
        'reference_date',
        'brl_price',
        config.exog_list
    )

    df_train_sku.index.freq = 'MS'
    df_test_sku.index.freq = 'MS'

    df_exog_train_sku.index.freq = 'MS'
    df_exog_test_sku.index.freq = 'MS'

    dict_modelos_sku = mg.criar_modelos_fipe(
        df_train_sku,
        df_test_sku,
        df_train_prophet_sku,
        df_test_prophet_sku,
        df_exog_train_sku,
        df_exog_test_sku,
        config.n_trials_skus,
        config.metrica_erro,
        config.tolerancia_fipe,
        'brl_price'
    )

    forecast_sku, modelo_escolhido_sku = utils.selecionar_modelo_fipe(
        df_train_sku,
        df_test_sku,
        df_train_prophet_sku,
        df_test_prophet_sku,
        df_exog_train_sku,
        df_exog_test_sku,
        df_exog_previsao,
        config.horizonte_previsao,
        'brl_price',
        **dict_modelos_sku
    )

    previsoes_por_sku[sku] = forecast_sku
    modelo_vencedor_por_sku[sku] = modelo_escolhido_sku

c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
15:51:25 - cmdstanpy - INFO - Chain [1] start processing
15:51:26 - cmdstanpy - INFO - Chain [1] done processing
15:51:26 - cmdstanpy - INFO - Chain [1] start processing
15:51:27 - cmdstanpy - INFO - Chain [1] done processing
15:51:27 - cmdstanpy 

255.76525297619048


c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters 

439.2392113095238


c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
15:52:01 - cmdstanpy - INFO - Chain [1] start processing
15:52:01 - cmdstanpy - INFO - Chain [1] done processing
15:52:01 - cmdstanpy - INFO - Chain [1] start processing
15:52:01 - cmdstanpy - INFO - Chain [1] done processing
15:52:01 - cmdstanpy - INFO - Chain [1] start processing
15:52:01 - cmdstanpy - INFO - Chain [1] done processing
15:52:02 - cmdstanpy - INFO - Chain [1] start processing
15:52:02 - cmdstanpy - INFO - Chain [1] done processing
15:52:02 - cmdstanpy - INFO - Chain [1] start 

5589.236607142857


c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
15:52:04 - cmdstanpy - INFO - Chain [1] start processing
15:52:04 - cmdstanpy - INFO - Chain [1] done processing
15:52:04 - cmdstanpy - INFO - Chain [1] start processing
15:52:05 - cmdstanpy - INFO - Chain [1] done processing
15:52:05 - cmdstanpy - INFO - Chain [1] start processing
15:52:05 - cmdstanpy - INFO - Chain [1] done processing
15:52:05 - cmdstanpy - INFO - Chain [1] start processing
15:52:06 - cmdstanpy - INFO - Cha

4688.558035714285


c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Vitor Rodrigues\Desktop\trabalho_icd\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check 

317.7167038690476


In [21]:
previsoes_por_sku

{100: 0    66161.468750
 1    66953.984375
 2    66151.367188
 Name: brl_price, dtype: float32,
 1832: 0    65228.906250
 1    65141.183594
 2    65285.316406
 Name: brl_price, dtype: float32,
 2134: 0     99588.062500
 1    100491.289062
 2    102213.796875
 Name: brl_price, dtype: float32,
 5112: 0    92834.242188
 1    92806.585938
 2    94309.945312
 Name: brl_price, dtype: float32,
 7023: 0    42585.183594
 1    42230.773438
 2    41956.453125
 Name: brl_price, dtype: float32}

In [22]:
modelo_vencedor_por_sku

{100: 'XGBOOST',
 1832: 'XGBOOST',
 2134: 'XGBOOST',
 5112: 'XGBOOST',
 7023: 'XGBOOST'}